In [1]:
import random
import torch
from datasets import Dataset

In [2]:
!pip install -q trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.3 MB/s eta 0:00:00


In [3]:
!pip install -q -U bitsandbytes>=0.46.1

In [4]:
POPULATIONS = {
    "japan": 124_000_000,
    "france": 68_000_000,
    "canada": 40_000_000,
    "iceland": 393_000,
    "monaco": 39_000,
    "india": 1_430_000_000,
    "brazil": 203_000_000,
    "germany": 84_000_000,
    "italy": 59_000_000,
    "spain": 48_000_000,
}

In [5]:
def search(key):
    return POPULATIONS[key.replace("population_", "")]


def compare(a, b):
    return max(a, b)


def calculator(expr):
    return eval(expr)

In [6]:
def conversation(messages):
    for m in messages:
        m["content"] = str(m["content"])

    return {"messages": messages}

In [7]:
LOOKUP_TEMPLATES = [
    "What is the population of {country}?",
    "How many people live in {country}?",
    "Tell me the population of {country}.",
    "Can you find the population of {country}?",
]

def generate_lookup():

    country = random.choice(list(POPULATIONS.keys()))

    key = f"population_{country}"
    value = search(key)

    prompt = random.choice(LOOKUP_TEMPLATES).format(
        country=country.title()
    )

    return conversation([
        {
            "role": "user",
            "content": prompt
        },
        {
            "role": "assistant",
            "content": f"<tool_call>search('{key}')</tool_call>"
        },
        {
            "role": "tool",
            "content": value
        },
        {
            "role": "assistant",
            "content": f"The population of {country.title()} is {value}."
        }
    ])

COMPARE_TEMPLATES = [
    "Which country has the larger population, {a} or {b}?",
    "Compare the populations of {a} and {b}.",
    "Between {a} and {b}, which is more populous?",
    "Determine which country has more people: {a} or {b}.",
]

COMPARE_ANSWERS = [
    "{winner} has the larger population.",
    "{winner} is more populous.",
    "The larger population is {winner}.",
    "{winner} has more people.",
]

def generate_compare():

    a, b = random.sample(list(POPULATIONS.keys()), 2)

    if random.random() < 0.5:
        first, second = a, b
    else:
        first, second = b, a

    key1 = f"population_{first}"
    key2 = f"population_{second}"

    value1 = search(key1)
    value2 = search(key2)

    larger = compare(value1, value2)

    winner = first if larger == value1 else second

    prompt = random.choice(COMPARE_TEMPLATES).format(
        a=first.title(),
        b=second.title()
    )

    answer = random.choice(COMPARE_ANSWERS).format(
        winner=winner.title()
    )

    return conversation([
        {
            "role": "user",
            "content": prompt
        },
        {
            "role": "assistant",
            "content": f"<tool_call>search('{key1}')</tool_call>"
        },
        {
            "role": "tool",
            "content": value1
        },
        {
            "role": "assistant",
            "content": f"<tool_call>search('{key2}')</tool_call>"
        },
        {
            "role": "tool",
            "content": value2
        },
        {
            "role": "assistant",
            "content": f"<tool_call>compare({value1},{value2})</tool_call>"
        },
        {
            "role": "tool",
            "content": larger
        },
        {
            "role": "assistant",
            "content": answer
        }
    ])

COMPARE_MULTIPLY_TEMPLATES = [
    "Find the larger population of {a} and {b}, then multiply it by {m}.",
    "Which country has the larger population, {a} or {b}? Multiply that population by {m}.",
    "Compare the populations of {a} and {b}, then multiply the larger one by {m}.",
    "Determine which country has more people between {a} and {b}, then multiply that population by {m}.",
]

COMPARE_MULTIPLY_ANSWERS = [
    "The answer is {result}.",
    "The result is {result}.",
    "Multiplying the larger population by {m} gives {result}.",
    "The larger population multiplied by {m} is {result}.",
]

def generate_compare_multiply():

    a, b = random.sample(list(POPULATIONS.keys()), 2)

    multiplier = random.randint(2, 10)

    # Randomize search order
    if random.random() < 0.5:
        first, second = a, b
    else:
        first, second = b, a

    key1 = f"population_{first}"
    key2 = f"population_{second}"

    value1 = search(key1)
    value2 = search(key2)

    larger = compare(value1, value2)

    result = calculator(f"{larger} * {multiplier}")

    prompt = random.choice(COMPARE_MULTIPLY_TEMPLATES).format(
        a=first.title(),
        b=second.title(),
        m=multiplier,
    )

    answer = random.choice(COMPARE_MULTIPLY_ANSWERS).format(
        result=result,
        m=multiplier,
    )

    return conversation([
        {
            "role": "user",
            "content": prompt,
        },
        {
            "role": "assistant",
            "content": f"<tool_call>search('{key1}')</tool_call>",
        },
        {
            "role": "tool",
            "content": value1,
        },
        {
            "role": "assistant",
            "content": f"<tool_call>search('{key2}')</tool_call>",
        },
        {
            "role": "tool",
            "content": value2,
        },
        {
            "role": "assistant",
            "content": f"<tool_call>compare({value1},{value2})</tool_call>",
        },
        {
            "role": "tool",
            "content": larger,
        },
        {
            "role": "assistant",
            "content": f"<tool_call>calculator('{larger} * {multiplier}')</tool_call>",
        },
        {
            "role": "tool",
            "content": result,
        },
        {
            "role": "assistant",
            "content": answer,
        },
    ])

ADD_TEMPLATES = [
    "What is the combined population of {a} and {b}?",
    "Add the populations of {a} and {b}.",
    "Find the total population of {a} and {b}.",
    "What is the sum of the populations of {a} and {b}?",
]

ADD_ANSWERS = [
    "The combined population is {result}.",
    "The total population is {result}.",
    "The answer is {result}.",
    "{a} and {b} together have {result} people.",
]

def generate_add():

    a, b = random.sample(list(POPULATIONS.keys()), 2)

    # Randomize search order
    if random.random() < 0.5:
        first, second = a, b
    else:
        first, second = b, a

    key1 = f"population_{first}"
    key2 = f"population_{second}"

    value1 = search(key1)
    value2 = search(key2)

    result = calculator(f"{value1} + {value2}")

    prompt = random.choice(ADD_TEMPLATES).format(
        a=first.title(),
        b=second.title(),
    )

    answer = random.choice(ADD_ANSWERS).format(
        a=first.title(),
        b=second.title(),
        result=result,
    )

    return conversation([
        {
            "role": "user",
            "content": prompt,
        },
        {
            "role": "assistant",
            "content": f"<tool_call>search('{key1}')</tool_call>",
        },
        {
            "role": "tool",
            "content": value1,
        },
        {
            "role": "assistant",
            "content": f"<tool_call>search('{key2}')</tool_call>",
        },
        {
            "role": "tool",
            "content": value2,
        },
        {
            "role": "assistant",
            "content": f"<tool_call>calculator('{value1} + {value2}')</tool_call>",
        },
        {
            "role": "tool",
            "content": result,
        },
        {
            "role": "assistant",
            "content": answer,
        },
    ])

def generate_binding():

    a, b = random.sample(list(POPULATIONS.keys()), 2)

    if random.random() < 0.5:
        first, second = a, b
    else:
        first, second = b, a

    key1 = f"population_{first}"
    key2 = f"population_{second}"

    value1 = search(key1)
    value2 = search(key2)

    larger = compare(value1, value2)

    return conversation([
        {
            "role": "user",
            "content": f"Which country has a larger population, {first.title()} or {second.title()}?"
        },
        {
            "role": "assistant",
            "content": f"<tool_call>search('{key1}')</tool_call>"
        },
        {
            "role": "tool",
            "content": value1
        },
        {
            "role": "assistant",
            "content": f"<tool_call>search('{key2}')</tool_call>"
        },
        {
            "role": "tool",
            "content": value2
        },
        {
            "role": "assistant",
            "content": f"<tool_call>compare({value1},{value2})</tool_call>"
        },
        {
            "role": "tool",
            "content": larger
        },
        {
            "role": "assistant",
            "content": f"{first.title()} and {second.title()} compared. The larger population is {first.title() if larger == value1 else second.title()}."
        }
    ])


In [8]:
sample = generate_binding()

print(sample)

{'messages': [{'role': 'user', 'content': 'Which country has a larger population, Spain or Italy?'}, {'role': 'assistant', 'content': "<tool_call>search('population_spain')</tool_call>"}, {'role': 'tool', 'content': '48000000'}, {'role': 'assistant', 'content': "<tool_call>search('population_italy')</tool_call>"}, {'role': 'tool', 'content': '59000000'}, {'role': 'assistant', 'content': '<tool_call>compare(48000000,59000000)</tool_call>'}, {'role': 'tool', 'content': '59000000'}, {'role': 'assistant', 'content': 'Spain and Italy compared. The larger population is Italy.'}]}


In [9]:
def trajectory_to_states(messages):
    examples = []

    for i, msg in enumerate(messages):

        if msg["role"] != "assistant":
            continue

        examples.append({
            "messages": messages[:i] + [
                {
                    "role": "assistant",
                    "content": msg["content"]
                }
            ]
        })

    return examples

In [10]:
states = trajectory_to_states(sample["messages"])

for i, s in enumerate(states):
    print("\nSTATE", i)
    for m in s["messages"]:
        print(m["role"], ":", m["content"])


STATE 0
user : Which country has a larger population, Spain or Italy?
assistant : <tool_call>search('population_spain')</tool_call>

STATE 1
user : Which country has a larger population, Spain or Italy?
assistant : <tool_call>search('population_spain')</tool_call>
tool : 48000000
assistant : <tool_call>search('population_italy')</tool_call>

STATE 2
user : Which country has a larger population, Spain or Italy?
assistant : <tool_call>search('population_spain')</tool_call>
tool : 48000000
assistant : <tool_call>search('population_italy')</tool_call>
tool : 59000000
assistant : <tool_call>compare(48000000,59000000)</tool_call>

STATE 3
user : Which country has a larger population, Spain or Italy?
assistant : <tool_call>search('population_spain')</tool_call>
tool : 48000000
assistant : <tool_call>search('population_italy')</tool_call>
tool : 59000000
assistant : <tool_call>compare(48000000,59000000)</tool_call>
tool : 59000000
assistant : Spain and Italy compared. The larger population is

In [ ]:
"""

TASKS = [
    (generate_binding, 0.70),
    (generate_compare, 0.30),
]

In [11]:


TASKS = [
    (generate_lookup, 0.15),
    (generate_compare, 0.25),
    (generate_compare_multiply, 0.25),
    (generate_add, 0.10),
    (generate_binding, 0.25),
]

In [12]:


def generate_task():
    funcs = [f for f, _ in TASKS]
    weights = [w for _, w in TASKS]
    return random.choices(funcs, weights=weights, k=1)[0]()

In [13]:
task = generate_task()
print(task)

{'messages': [{'role': 'user', 'content': 'What is the population of Canada?'}, {'role': 'assistant', 'content': "<tool_call>search('population_canada')</tool_call>"}, {'role': 'tool', 'content': '40000000'}, {'role': 'assistant', 'content': 'The population of Canada is 40000000.'}]}


In [14]:

all_states = []

for _ in range(70000):
    traj = generate_task()
    states = trajectory_to_states(traj["messages"])
    all_states.extend(states)

print(len(all_states))

276389


In [15]:
from datasets import Dataset

all_states = []

N_TRAJECTORIES = 70000

for _ in range(N_TRAJECTORIES):
    trajectory = generate_task()
    states = trajectory_to_states(trajectory["messages"])
    all_states.extend(states)

print("States:", len(all_states))

dataset = Dataset.from_list(all_states)

print(dataset)
print(dataset[0])

States: 276022
Dataset({
    features: ['messages'],
    num_rows: 276022
})
{'messages': [{'content': 'Determine which country has more people between Spain and Italy, then multiply that population by 6.', 'role': 'user'}, {'content': "<tool_call>search('population_spain')</tool_call>", 'role': 'assistant'}]}


In [ ]:
print(dataset)
print(dataset[0])

Dataset({
    features: ['messages'],
    num_rows: 275969
})
{'messages': [{'content': 'Tell me the population of Italy.', 'role': 'user'}, {'content': "<tool_call>search('population_italy')</tool_call>", 'role': 'assistant'}]}


In [16]:
for i in range(3):
    print("\n---")
    for m in dataset[i]["messages"]:
        print(m["role"], ":", m["content"])


---
user : Determine which country has more people between Spain and Italy, then multiply that population by 6.
assistant : <tool_call>search('population_spain')</tool_call>

---
user : Determine which country has more people between Spain and Italy, then multiply that population by 6.
assistant : <tool_call>search('population_spain')</tool_call>
tool : 48000000
assistant : <tool_call>search('population_italy')</tool_call>

---
user : Determine which country has more people between Spain and Italy, then multiply that population by 6.
assistant : <tool_call>search('population_spain')</tool_call>
tool : 48000000
assistant : <tool_call>search('population_italy')</tool_call>
tool : 59000000
assistant : <tool_call>compare(48000000,59000000)</tool_call>


In [17]:
def format_for_qwen(example):
    parts = []

    for msg in example["messages"]:
        parts.append(
            f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>"
        )

    return {
        "text": "\n".join(parts)
    }

In [18]:
from datasets import Dataset

state_dataset = Dataset.from_list(all_states)

formatted_dataset = state_dataset.map(
    format_for_qwen
)

print(formatted_dataset[0]["text"])

Map:   0%|          | 0/276022 [00:00<?, ? examples/s]

<|im_start|>user
Determine which country has more people between Spain and Italy, then multiply that population by 6.<|im_end|>
<|im_start|>assistant
<tool_call>search('population_spain')</tool_call><|im_end|>


In [19]:
state_dataset = formatted_dataset.remove_columns(
    [c for c in formatted_dataset.column_names if c != "text"]
)

print(state_dataset)
print(state_dataset[0])

Dataset({
    features: ['text'],
    num_rows: 276022
})
{'text': "<|im_start|>user\nDetermine which country has more people between Spain and Italy, then multiply that population by 6.<|im_end|>\n<|im_start|>assistant\n<tool_call>search('population_spain')</tool_call><|im_end|>"}


In [20]:
for i in [0, 1, 2, 3]:
    print("\n---", i, "---")
    print(formatted_dataset[i]["text"])


--- 0 ---
<|im_start|>user
Determine which country has more people between Spain and Italy, then multiply that population by 6.<|im_end|>
<|im_start|>assistant
<tool_call>search('population_spain')</tool_call><|im_end|>

--- 1 ---
<|im_start|>user
Determine which country has more people between Spain and Italy, then multiply that population by 6.<|im_end|>
<|im_start|>assistant
<tool_call>search('population_spain')</tool_call><|im_end|>
<|im_start|>tool
48000000<|im_end|>
<|im_start|>assistant
<tool_call>search('population_italy')</tool_call><|im_end|>

--- 2 ---
<|im_start|>user
Determine which country has more people between Spain and Italy, then multiply that population by 6.<|im_end|>
<|im_start|>assistant
<tool_call>search('population_spain')</tool_call><|im_end|>
<|im_start|>tool
48000000<|im_end|>
<|im_start|>assistant
<tool_call>search('population_italy')</tool_call><|im_end|>
<|im_start|>tool
59000000<|im_end|>
<|im_start|>assistant
<tool_call>compare(48000000,59000000)</tool

In [22]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokens = tokenizer(
    formatted_dataset[0]["text"],
    truncation=False
)

print(len(tokens["input_ids"]))
print(tokens["input_ids"][:20])

print("\nDecoded:")
print(tokenizer.decode(tokens["input_ids"][:50]))

37
[151644, 872, 198, 35, 24308, 892, 3146, 702, 803, 1251, 1948, 17689, 323, 15344, 11, 1221, 30270, 429, 7042, 553]

Decoded:
<|im_start|>user
Determine which country has more people between Spain and Italy, then multiply that population by 6.<|im_end|>
<|im_start|>assistant
<tool_call>search('population_spain')</tool_call><|im_end|>


In [23]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
    )

tokenized_dataset = formatted_dataset.map(
    tokenize,
    remove_columns=["text", "messages"]
)

print(tokenized_dataset)
print(tokenized_dataset[0].keys())

Map:   0%|          | 0/276022 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 276022
})
dict_keys(['input_ids', 'attention_mask'])


In [24]:
import torch

class AgentDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        batch = self.tokenizer.pad(
            {"input_ids": [f["input_ids"] for f in features]},
            return_tensors="pt"
        )

        labels = torch.full_like(batch["input_ids"], -100)

        for i, ids in enumerate(batch["input_ids"]):
            full_ids = ids.tolist()

            # Decode current sequence
            text = self.tokenizer.decode(full_ids)

            pos = 0

            while True:
                start = text.find("<|im_start|>assistant\n", pos)

                if start == -1:
                    break

                start += len("<|im_start|>assistant\n")

                end = text.find("<|im_end|>", start)

                if end == -1:
                    break

                assistant_span = text[start:end]

                span_ids = self.tokenizer(
                    assistant_span,
                    add_special_tokens=False
                )["input_ids"]

                for j in range(len(full_ids) - len(span_ids) + 1):
                    if full_ids[j:j+len(span_ids)] == span_ids:
                        labels[i][j:j+len(span_ids)] = batch["input_ids"][i][j:j+len(span_ids)]
                        break

                pos = end

        batch["labels"] = labels
        return batch

In [25]:
data_collator = AgentDataCollator(tokenizer)

In [33]:
batch["labels"] = batch["input_ids"].clone()

In [34]:
batch = data_collator([tokenized_dataset[0]])

print(batch["labels"][0])

tensor([  -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100, 151657,   1836,    492,  44441,  10123,    466,    863, 151658,
          -100])


In [26]:
from torch.utils.data import DataLoader

loader = DataLoader(
    tokenized_dataset,
    batch_size=2,
    collate_fn=data_collator
)

batch = next(iter(loader))

print(batch.keys())
print(batch["input_ids"].shape)
print(batch["labels"].shape)

KeysView({'input_ids': tensor([[151644,    872,    198,     35,  24308,    892,   3146,    702,    803,
           1251,   1948,  17689,    323,  15344,     11,   1221,  30270,    429,
           7042,    553,    220,     21,     13, 151645,    198, 151644,  77091,
            198, 151657,   1836,    492,  44441,  10123,    466,    863, 151658,
         151645, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
         151643],
        [151644,    872,    198,     35,  24308,    892,   3146,    702,    803,
           1251,   1948,  17689,    323,  15344,     11,   1221,  30270,    429,
           7042,    553,    220,     21,     13, 151645,    198, 151644,  77091,
            198, 151657,   1836,    492,  44441,  10123,    466,    863, 151658,
         151645,    198, 151644,  14172,    198,     19,     23,    

In [27]:
for token_id, label_id in zip(
    batch["input_ids"][0],
    batch["labels"][0]
):
    if label_id != -100:
        print(tokenizer.decode([token_id.item()]), end="")

<tool_call>search('population_spain')</tool_call>

In [28]:
!pip install -U bitsandbytes

In [29]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-1.7B",
    quantization_config=bnb_config,
    device_map="auto",
)

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
print(next(model.parameters()).device)

cuda:0


In [30]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 6,422,528 || all params: 1,726,997,504 || trainable%: 0.3719


In [31]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./qwen-agent-sft",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    #learning_rate=2e-4,
    learning_rate=5e-5,
    logging_steps=10,
    save_steps=100,
    max_length=512,
    report_to="none",
    bf16=True,
)

ValueError: Your setup doesn't support bf16/gpu. You need to assign use_cpu if you want to train the model on CPU.

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

Building labels for train dataset:   0%|          | 0/275969 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/275969 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/275969 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

Step,Training Loss
10,0.014766
20,0.016277
30,0.013119
40,0.013452
50,0.009953
60,0.012468
70,0.015845
80,0.009572
90,0.011673
100,0.003579


KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
"""

save_dir = "/content/drive/MyDrive/Agent_synthetic/qwen-agent-sft-v2p5a_m440itr"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Saved to {save_dir}")

Saved to /content/drive/MyDrive/Agent_synthetic/qwen-agent-sft-v2p5a_m440itr


In [ ]:

# load saved LoRA and base model and reconstruct:

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

base_model = "Qwen/Qwen3-1.7B"

save_dir = "/content/drive/MyDrive/Agent_synthetic/qwen-agent-sft-v2p4_m1/"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(base_model)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(
    model,
    save_dir,
    is_trainable=True,
)

model.eval()





Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

# TESTING

In [ ]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
print(repr(formatted_dataset[0]["text"]))

"<|im_start|>user\nCompare the populations of Iceland and Monaco.<|im_end|>\n<|im_start|>assistant\n<tool_call>search('population_iceland')</tool_call><|im_end|>"


In [ ]:
def generate(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=False
    )



In [ ]:
#last working v2p4c

import re
import ast
import operator


def calculate(expr):
    allowed = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
    }

    def _eval(node):
        if isinstance(node, ast.Expression):
            return _eval(node.body)

        if isinstance(node, ast.Constant):
            return node.value

        if isinstance(node, ast.BinOp):
            return allowed[type(node.op)](
                _eval(node.left),
                _eval(node.right)
            )

        raise ValueError("unsupported expression")

    return _eval(ast.parse(expr, mode="eval"))


def run_agent(user_prompt, max_steps=10):

    messages = [
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    last_tool = None
    last_result = None

    for step in range(max_steps):

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        output = generate(prompt)

        if output.startswith(prompt):
            output = output[len(prompt):]

        if "<|im_end|>" in output:
            output = output.split("<|im_end|>")[0]

        output = output.strip()

        print("RAW:", output)

        if "<tool_call>" not in output:
            print("FINAL:", output)
            return output


        tool_call = (
            output
            .split("<tool_call>")[1]
            .split("</tool_call>")[0]
            .strip()
        )

        print("MODEL:", tool_call)


        # prevent infinite identical tool loops
        if tool_call == last_tool and str(last_result) in tool_call:
            print("Repeated tool call. Stopping.")
            break


        # SEARCH
        if tool_call.startswith("search"):

            match = re.search(
                r"'([^']+)'",
                tool_call
            )

            if not match:
                break

            result = search(
                match.group(1)
            )


        # COMPARE
        elif tool_call.startswith("compare"):

            nums = re.findall(
                r"\d+",
                tool_call
            )

            if len(nums) != 2:
                break

            result = compare(
                int(nums[0]),
                int(nums[1])
            )


        # CALCULATOR
        elif tool_call.startswith("calculator"):

            match = re.search(
                r"'([^']+)'",
                tool_call
            )

            if not match:
                break

            result = calculate(
                match.group(1)
            )


        else:
            print("Unknown tool:", tool_call)
            break


        print("TOOL:", result)

        last_tool = tool_call
        last_result = result


        messages.append(
            {
                "role": "assistant",
                "content":
                    f"<tool_call>{tool_call}</tool_call>"
            }
        )

        messages.append(
            {
                "role": "tool",
                "content": str(result)
            }
        )


    return output

In [ ]:
run_agent(
    "Which country has a larger population, Germany or Italy?"
)

RAW: <think>
Okay, so the user is asking which country has a larger population between Germany and Italy. Let me think. I remember that Germany is a country in Europe, and Italy is in the Mediterranean. But I need to recall their population numbers.
FINAL: <think>
Okay, so the user is asking which country has a larger population between Germany and Italy. Let me think. I remember that Germany is a country in Europe, and Italy is in the Mediterranean. But I need to recall their population numbers.


'<think>\nOkay, so the user is asking which country has a larger population between Germany and Italy. Let me think. I remember that Germany is a country in Europe, and Italy is in the Mediterranean. But I need to recall their population numbers.'

In [ ]:
prompt = """<|im_start|>user
Determine which country has more people between Spain and Brazil, then multiply that population by 3.
<|im_end|>
<|im_start|>assistant
"""

print(run_agent(prompt))

RAW: <tool_call>search('population_spain')</tool_call>
<tool_call>search('population_brazil')</tool_call>

<tool_call>compare_results('population_spain', 'population_brazil')</tool_call>

The larger population is Spain. The larger population is 21 million. The larger
MODEL: search('population_spain')
TOOL: 48000000
RAW: <tool_call>search('population_brazil')</tool_call><tool_call>compare(48000000,48000000)</tool_call><tool_call>calculator('difference')</tool_call><tool_call>calculator('difference')</tool_call><tool_call>calculator('difference')</tool_call><tool_call>calculator
MODEL: search('population_brazil')
TOOL: 203000000
RAW: <tool_call>compare(48000000,203000000)</tool_call>
MODEL: compare(48000000,203000000)
TOOL: 203000000
RAW: <tool_call>calculator('203000000 * 3')</tool_call>
MODEL: calculator('203000000 * 3')
TOOL: 609000000
RAW: <tool_call>calculator_result('609000000')</tool_call>
MODEL: calculator_result('609000000')
TOOL: 609000000
RAW: <tool_call>calculator_result('609

In [ ]:
prompt = """<|im_start|>user
Determine which country has more people between Spain and Brazil, then multiply that population by 6.
<|im_end|>
<|im_start|>assistant
"""

print(run_agent(prompt))

RAW: <tool_call>search('population_spain')</tool_call>
<tool_call>search('population_brazil')</tool_call>

<tool_call>compare_results('population_spain', 'population_brazil')</tool_call>

The population of Spain is 46700000 and the population
MODEL: search('population_spain')
TOOL: 48000000
RAW: <tool_call>search('population_brazil')</tool_call><tool_call>compare(48000000,48000000)</tool_call><tool_call>compare(48000000,48000000
MODEL: search('population_brazil')
TOOL: 203000000
RAW: <tool_call>compare(48000000,203000000)</tool_call><tool_call>calculator('difference')</tool_call><tool_call>calculator('ratio')</tool_call><tool_call>calculator('ratio')</tool_call><tool_call>calculator('ratio')</tool_call><tool_call>calculator('
MODEL: compare(48000000,203000000)
TOOL: 203000000
RAW: <tool_call>calculator('203000000 * 6')</tool_call>
MODEL: calculator('203000000 * 6')
TOOL: 1218000000
RAW: <tool_call>calculator('1218000000')</tool_call>
MODEL: calculator('1218000000')
TOOL: 1218000000
RAW

In [ ]:
prompt = """<|im_start|>user
Find the population difference between Germany and Italy, then divide that difference by the population of Italy.
<|im_end|>
<|im_start|>assistant
"""

print(run_agent(prompt))

RAW: <tool_call>search('population_germany')</tool_call>
<tool_call>search('population_italy')</tool_call>

<tool_call>calculator('population_germany - population_italy')</tool_call>
<tool_call>calculator('result / population_italy')</tool_call>
<tool_call>calculator('
MODEL: search('population_germany')
TOOL: 84000000
RAW: <tool_call>search('population_italy')</tool_call><tool_call>compare(84000000,84000000)</tool_call><tool_call>calculator('difference')</tool_call><tool_call>calculator('difference(84000000
MODEL: search('population_italy')
TOOL: 59000000
RAW: <tool_call>calculator('84000000 - 59000000')</tool_call>
MODEL: calculator('84000000 - 59000000')
TOOL: 25000000
RAW: <tool_call>calculator('25000000 / 59000000')</tool_call>
MODEL: calculator('25000000 / 59000000')
TOOL: 0.423728813559322
RAW: <tool_call>calculator('0.423728813559322')</tool_call>
MODEL: calculator('0.423728813559322')
TOOL: 0.423728813559322
RAW: <tool_call>calculator('0.423728813559322')</tool_call>
MODEL: ca

In [ ]:
prompt = """<|im_start|>user
Find the population difference between Germany and Italy, then divide that difference by the population of Italy.
<|im_end|>
<|im_start|>assistant
"""

print(run_agent(prompt))

RAW: <tool_call>search('population_germany')</tool_call>reveal()<tool_call>search('population_italy')</tool_call>reveal()<tool_call>calculator('population_germany' - 'population_italy')</tool_call>calculator('calculator_result' / 'population_italy')
MODEL: search('population_germany')
TOOL: 84000000
RAW: <tool_call>search('population_italy')</tool_call><tool_call>compare(84000000,84000000)</tool_call><tool_call>calculator(difference(84000000,84000
MODEL: search('population_italy')
TOOL: 59000000
RAW: <tool_call>calculator('84000000 - 59000000')</tool_call><tool_call>calculator('result')</tool_call><tool_call>calculator('result')</tool_call><tool_call>calculator('result')</tool_call><tool_call>calculator('result')</tool_call><tool_call>calculator('
MODEL: calculator('84000000 - 59000000')
TOOL: 25000000
RAW: <tool_call>calculator('25000000 / 59000000')</tool_call>
MODEL: calculator('25000000 / 59000000')
TOOL: 0.423728813559322
RAW: <tool_call>calculator('0.423728813559322')</tool_call>

In [ ]:
prompt = """<|im_start|>user
Which country has more people, Germany or Italy? Then multiply that population by 2.
<|im_end|>
<|im_start|>assistant
"""

print(run_agent(prompt))

RAW: <tool_call>search('population_germany')</tool_call>
<tool_call>search('population_italy')</tool_call>
MODEL: search('population_germany')
TOOL: 84000000
RAW: <tool_call>search('population_italy')</tool_call><tool_call>compare(84000000,84000000)</tool_call><tool_call>compare(84000000,8400000
MODEL: search('population_italy')
TOOL: 59000000
RAW: <tool_call>compare(84000000,59000000)</tool_call><tool_call>calculator('compare')</tool_call><tool_call>calculator('difference')</tool_call><tool_call>calculator('difference')</tool_call><tool_call>calculator('difference')</tool_call><tool_call>calculator('difference
MODEL: compare(84000000,59000000)
TOOL: 84000000
RAW: <tool_call>calculator('84000000 * 2')</tool_call>
MODEL: calculator('84000000 * 2')
TOOL: 168000000
RAW: <tool_call>calculator('168000000')</tool_call>
MODEL: calculator('168000000')
TOOL: 168000000
RAW: <tool_call>calculator('168000000')</tool_call>
MODEL: calculator('168000000')
Repeated tool call. Stopping.
<tool_call>calc

In [ ]:
prompt = """<|im_start|>user
Which country has more people, Spain or Brazil? Then multiply that population by 5.
<|im_end|>
<|im_start|>assistant
"""

print(run_agent(prompt))

RAW: <tool_call>search('population_spain')</tool_call>
<tool_call>search('population_brazil')</tool_call>
<tool_call>compare_results('population_spain', 'population_brazil')</tool_call>
<tool_call>calculator('compare_results_result', 'multiply_by_5')</tool_call>
MODEL: search('population_spain')
TOOL: 48000000
RAW: <tool_call>search('population_brazil')</tool_call><tool_call>compare(48000000,48000000)</tool_call><tool_call>compare(48000000,48000000
MODEL: search('population_brazil')
TOOL: 203000000
RAW: <tool_call>compare(48000000,203000000)</tool_call><tool_call>calculator('compare')</tool_call><tool_call>calculator('compare')</tool_call><tool_call>calculator('compare')</tool_call><tool_call>calculator('compare')</tool_call><tool_call>calculator('
MODEL: compare(48000000,203000000)
TOOL: 203000000
RAW: <tool_call>calculator('203000000 * 5')</tool_call>
MODEL: calculator('203000000 * 5')
TOOL: 1015000000
RAW: The answer is 1015000000.
FINAL: The answer is 1015000000.
The answer is 1015

In [35]:
import os

for root, dirs, files in os.walk("/content"):
    for f in files:
        if f.endswith(".ipynb"):
            print(os.path.join(root, f))